In [1]:
from pathlib import Path
file_path = Path('./data/federalist_papers.txt')

In [2]:
# I should probably write a code that checks if smth is smth 
# import requests
# r = requests.get('https://www.gutenberg.org/cache/epub/1404/pg1404.txt')
# with open(file_path , 'wb') as f:
#     f.write(r.content)

In [3]:
import spacy 
nlp = spacy.load('en_core_web_md')
nlp.add_pipe('sentencizer')

In [4]:
f = open(file_path, 'r', encoding='utf-8')

collection_paper = []
federalist_paper = []

for f_line in f:
    if f_line.startswith('*** END OF THE PROJECT GUTENBERG'):
        break
    if f_line.startswith('FEDERALIST No.'):
        federalist_paper = [] 
        collection_paper.append(federalist_paper)
    federalist_paper.append(f_line)


print(collection_paper[0])
print(len(collection_paper))
f.close()

['FEDERALIST No. 1\n', '\n', 'General Introduction\n', '\n', 'For the Independent Journal. Saturday, October 27, 1787\n', '\n', '\n', 'HAMILTON\n', '\n', 'To the People of the State of New York:\n', '\n', 'AFTER an unequivocal experience of the inefficacy of the subsisting\n', 'federal government, you are called upon to deliberate on a new\n', 'Constitution for the United States of America. The subject speaks its\n', 'own importance; comprehending in its consequences nothing less than the\n', 'existence of the UNION, the safety and welfare of the parts of which it\n', 'is composed, the fate of an empire in many respects the most interesting\n', 'in the world. It has been frequently remarked that it seems to have been\n', 'reserved to the people of this country, by their conduct and example,\n', 'to decide the important question, whether societies of men are really\n', 'capable or not of establishing good government from reflection and\n', 'choice, or whether they are forever destined t

In [5]:
import re 

collection_paper = [[line.strip() for line in paper if line.strip()] for paper in collection_paper]

author_list = ('HAMILTON', 'JAY', 'MADISON')
collection_dict = []

for paper in collection_paper:
    paper_dict = {'paper_no': 0, 'supposed_author': '', 'content': [], 'reference': []} 

    state = 'CONTENT'
    for line in paper:
        if line.startswith('FEDERALIST No'):
            paper_dict['paper_no'] = int(re.search(r'\d+', line).group())
            continue
        if line.startswith(author_list):
            paper_dict['supposed_author'] = line
            continue
        if line == 'PUBLIUS':
            state = 'REFERENCE'
            continue
        if line.startswith('To the People of the State'):
            paper_dict['content'] = []
            continue
        if state == 'CONTENT':
            cleaned = re.sub(r'\(\d+\)', '', line)
            paper_dict['content'].append(cleaned)
        else:
            paper_dict['reference'].append(line)
    paper_dict['content'] = ' '.join(paper_dict['content'])
    collection_dict.append(paper_dict)

print(collection_dict[0])

{'paper_no': 1, 'supposed_author': 'HAMILTON', 'content': 'AFTER an unequivocal experience of the inefficacy of the subsisting federal government, you are called upon to deliberate on a new Constitution for the United States of America. The subject speaks its own importance; comprehending in its consequences nothing less than the existence of the UNION, the safety and welfare of the parts of which it is composed, the fate of an empire in many respects the most interesting in the world. It has been frequently remarked that it seems to have been reserved to the people of this country, by their conduct and example, to decide the important question, whether societies of men are really capable or not of establishing good government from reflection and choice, or whether they are forever destined to depend for their political constitutions on accident and force. If there be any truth in the remark, the crisis at which we are arrived may with propriety be regarded as the era in which that dec

In [6]:
for paper in collection_dict:
    print(paper['paper_no'], paper['supposed_author'])

1 HAMILTON
2 JAY
3 JAY
4 JAY
5 JAY
6 HAMILTON
7 HAMILTON
8 HAMILTON
9 HAMILTON
10 MADISON
11 HAMILTON
12 HAMILTON
13 HAMILTON
14 MADISON
15 HAMILTON
16 HAMILTON
17 HAMILTON
18 MADISON, with HAMILTON
19 MADISON, with HAMILTON
20 MADISON, with HAMILTON
21 HAMILTON
22 HAMILTON
23 HAMILTON
24 HAMILTON
25 HAMILTON
26 HAMILTON
27 HAMILTON
28 HAMILTON
29 HAMILTON
30 HAMILTON
31 HAMILTON
32 HAMILTON
33 HAMILTON
34 HAMILTON
35 HAMILTON
36 HAMILTON
37 MADISON
38 MADISON
39 MADISON
40 MADISON
41 MADISON
42 MADISON
43 MADISON
44 MADISON
45 MADISON
46 MADISON
47 MADISON
48 MADISON
49 MADISON
50 MADISON
51 MADISON
52 MADISON
53 MADISON
54 MADISON
55 MADISON
56 MADISON
57 MADISON
58 MADISON
59 HAMILTON
60 HAMILTON
61 HAMILTON
62 MADISON
63 MADISON
64 JAY
65 HAMILTON
66 HAMILTON
67 HAMILTON
68 HAMILTON
69 HAMILTON
70 HAMILTON
71 HAMILTON
72 HAMILTON
73 HAMILTON
74 HAMILTON
75 HAMILTON
76 HAMILTON
77 HAMILTON
78 HAMILTON
79 HAMILTON
80 HAMILTON
81 HAMILTON
82 HAMILTON
83 HAMILTON
84 HAMILTON
85 HAMILTO

In [7]:
accepted_essays = []
disputed_essays = []
joint_essays = []
disputed_no = [49,50,51,52,53,54,55,56,57,58, 62,63]

# Papers with joint should be dropped 
for paper in collection_dict: 
    if paper['paper_no'] in disputed_no:
        disputed_essays.append((paper['content'], paper['supposed_author']))
    else:
        if paper['supposed_author'] in author_list:
            accepted_essays.append((paper['content'], paper['supposed_author']))
        else:
            joint_essays.append((paper['content'], paper['supposed_author']))

print(len(accepted_essays))
print(len(disputed_essays))
print(len(joint_essays))


70
12
3


In [8]:
# feature functions:
def word_sentence(text):
    sents_per_text = len([sents for sents in text.sents])
    word_tokens = len([word.text for word in text])
    word_per_sent = float(word_tokens)/float(sents_per_text)
    return word_per_sent, sents_per_text

def type_counter(text): 
    type_count = {"ADJ": 0 , "ADP": 0, "ADV": 0, "AUX": 0, "CCONJ": 0,
    "DET": 0, "INTJ": 0, "NOUN": 0, "NUM": 0, "PART": 0,
    "PRON": 0, "PROPN": 0, "PUNCT": 0, "SCONJ": 0,
    "SYM": 0, "VERB": 0, "X": 0}

    for token in text: 
        if token.pos_ in type_count: 
            type_count[token.pos_] = type_count.get(token.pos_) + 1 
        else: 
            type_count[token.pos_] = 0
    return type_count


In [9]:
from collections import Counter

def common_words(text):
    top_n = 50
    ham_vocab = []
    mad_vocab = [] 
    jay_vocab = [] 
    for content, label in text: 
        tokens = [token.text.lower() for token in content]
        if label=='HAMILTON':
            ham_vocab.extend(tokens)
        elif label=='MADISON':
            mad_vocab.extend(tokens)
        elif label=='JAY':
            jay_vocab.extend(tokens)
    ham_top = {w for w, _ in Counter(ham_vocab).most_common(top_n)}
    mad_top = {w for w, _ in Counter(mad_vocab).most_common(top_n)}
    jay_top = {w for w, _ in Counter(jay_vocab).most_common(top_n)}
    return ham_top & mad_top & jay_top

In [10]:
def common_count(text, common_voc):
    common_voc_values = Counter(text)
    counts = {k: common_voc_values[k] for k in common_voc}
    return counts 

In [11]:
def type_counter(text, word_per_text): 
    type_count = {"ADJ": 0 , "ADP": 0, "ADV": 0, "AUX": 0, "CCONJ": 0,
    "DET": 0, "INTJ": 0, "NOUN": 0, "NUM": 0, "PART": 0,
    "PRON": 0, "PROPN": 0, "PUNCT": 0, "SCONJ": 0,
    "SYM": 0, "VERB": 0, "X": 0}

    for token in text: 
        if token.pos_ in type_count: 
            type_count[token.pos_] = type_count.get(token.pos_) + 1 
        else: 
            type_count[token.pos_] = 0
    return type_count

In [12]:
def word_count(content):
    word_count = {} 

    for word in content:
        word_lower = word.text.lower()
        if word_lower in word_count.keys(): 
            word_count[word_lower] = word_count.get(word_lower)+1
        else:
            word_count[word_lower] = 1
    return word_count

In [13]:
def noun_chunking(content):
    list_doc = [doc for doc, label in content]
    chunk_doc = {} 

    for doc in list_doc: 
        for chunk in doc.noun_chunks:
            chunk_append = chunk.text.lower()
            if len(chunk) > 1:
                if chunk_append in chunk_doc:
                    chunk_doc[chunk_append] = chunk_doc.get(chunk_append)+1
                else:
                    chunk_doc[chunk_append] = 1 
    chunk_list = {chunk for chunk, freq in Counter(chunk_doc).most_common(50)}
    return chunk_list



In [14]:
def chunk_count(content, chunk_list):
    present = Counter(
        chunk.text.lower()
        for chunk in content.noun_chunks
        if len(chunk) > 1
    )
    return {ch: present.get(ch, 0) for ch in chunk_list}

In [15]:
def suffix(content):
    suffix_doc = []
    doc_list = [doc for doc, label in content]
    for doc in doc_list:
        for word in doc:
            suffix_doc.append(str(word.suffix_).lower())
    suffix_count = Counter(suffix_doc)
    sorted_count = [suffix for suffix, freq in suffix_count.most_common(50)]
    return sorted_count


def suffix_count(content, suffix_list):
    count_doc = {}
    content_suffix = [str(word.suffix_) for word in content]
    count = Counter(content_suffix)
    for suffix in suffix_list:
        if suffix in count.keys():
            count_doc[suffix] = count.get(suffix)
        else:
            count_doc[suffix] = 0
    return count_doc

In [16]:
from spacy.lang.en import stop_words as stop_words
stop_words = stop_words.STOP_WORDS

def document_features(doc, word_per_text):
    features = []
    word_per_sent, sents_per_text = word_sentence(doc)

    features.append(word_per_sent)
    features.append(sents_per_text)

    pos_counts = type_counter(doc, word_per_text)
    for value in pos_counts.values():
        features.append(value / word_per_text)

    count = word_count(doc)
    for word in sorted(stop_words):
        features.append(count.get(word, 0))
    return features

In [17]:
def corpus_features(doc, common_voc, suffix_list, word_per_text, chunk_list):
    features = []

    count = word_count(doc)
    for word in common_voc:
        features.append(count.get(word, 0))

    count_doc = suffix_count(doc, suffix_list)        
    for suffix in count_doc.keys():
        features.append(count_doc.get(suffix)/word_per_text)

    chunk_doc = chunk_count(doc, chunk_list)
    for chunk in chunk_list:  
        features.append(chunk_doc.get(chunk, 0)) 
    return features

In [45]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report
from sklearn.base import clone
import numpy as np

content = [nlp(content) for content, label in accepted_essays]
labels = [label for content, label in accepted_essays]
y = np.array(labels)
loo = LeaveOneOut()
y_true, y_pred = [], []
text_clf = DecisionTreeClassifier(random_state=42)
list_importances = []


for train_idx, test_idx in loo.split(content):
    common_voc  = common_words([(content[i], labels[i]) for i in train_idx])
    suffix_list = suffix([(content[i], labels[i]) for i in train_idx])
    noun_chunk = noun_chunking([(content[i], labels[i]) for i in train_idx])
    
    X_train, X_test = [], []
    for i in train_idx:
        word_per_text = len([word for word in content[i]])
        f = document_features(content[i], word_per_text) + corpus_features(content[i], common_voc, suffix_list, word_per_text, noun_chunk)
        X_train.append(f)
    for i in test_idx:
        word_per_text = len(content[i])
        f = document_features(content[i], word_per_text) + corpus_features(content[i], common_voc, suffix_list, word_per_text, noun_chunk)
        X_test.append(f)
    X_train = np.array(X_train)
    X_test  = np.array(X_test)
    y_train = y[train_idx]
    y_test  = y[test_idx]

    model = clone(text_clf)
    model.fit(X_train, y_train)

    list_importances.append({'feature_importances': model.feature_importances_, 'common_voc': common_voc, 'suffix_list': suffix_list, 'noun_chunk': noun_chunk})
    
    predicted = model.predict(X_test)
    y_true.append(y_test[0])
    y_pred.append(predicted[0])

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    HAMILTON       0.96      0.90      0.93        51
         JAY       1.00      1.00      1.00         5
     MADISON       0.71      0.86      0.77        14

    accuracy                           0.90        70
   macro avg       0.89      0.92      0.90        70
weighted avg       0.91      0.90      0.90        70



In [37]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_true, y_pred))

[[46  0  5]
 [ 0  5  0]
 [ 2  0 12]]


In [47]:
import operator
from operator import itemgetter


In [62]:
first_split = list_importances[15]

def get_feature_ranking(split):
    feature_names = []
    feature_names.append('word_per_sent')
    feature_names.append('sents_per_text')

    type_count = {"ADJ": 0 , "ADP": 0, "ADV": 0, "AUX": 0, "CCONJ": 0,
    "DET": 0, "INTJ": 0, "NOUN": 0, "NUM": 0, "PART": 0,
    "PRON": 0, "PROPN": 0, "PUNCT": 0, "SCONJ": 0,
    "SYM": 0, "VERB": 0, "X": 0}
    for key, value in type_count.items():
        feature_names.append(str(key))
    for stopword in sorted(stop_words):
        feature_names.append(stopword)
    for word in split['common_voc']:
        feature_names.append(word)
    for suffix in split['suffix_list']:
        feature_names.append(suffix)
    for chunk in split['noun_chunk']:
        feature_names.append(chunk)

    sorted_names = sorted(zip(feature_names,split['feature_importances']), key=operator.itemgetter(1), reverse=True)
    print(len(feature_names), len(split['feature_importances']))
    return sorted_names

    
get_feature_ranking(first_split)

480 480


[('upon', np.float64(0.6311551770029278)),
 ('ies', np.float64(0.2387543252595156)),
 ('ten', np.float64(0.08499245852187037)),
 ('her', np.float64(0.04509803921568627)),
 ('word_per_sent', np.float64(0.0)),
 ('sents_per_text', np.float64(0.0)),
 ('ADJ', np.float64(0.0)),
 ('ADP', np.float64(0.0)),
 ('ADV', np.float64(0.0)),
 ('AUX', np.float64(0.0)),
 ('CCONJ', np.float64(0.0)),
 ('DET', np.float64(0.0)),
 ('INTJ', np.float64(0.0)),
 ('NOUN', np.float64(0.0)),
 ('NUM', np.float64(0.0)),
 ('PART', np.float64(0.0)),
 ('PRON', np.float64(0.0)),
 ('PROPN', np.float64(0.0)),
 ('PUNCT', np.float64(0.0)),
 ('SCONJ', np.float64(0.0)),
 ('SYM', np.float64(0.0)),
 ('VERB', np.float64(0.0)),
 ('X', np.float64(0.0)),
 ("'d", np.float64(0.0)),
 ("'ll", np.float64(0.0)),
 ("'m", np.float64(0.0)),
 ("'re", np.float64(0.0)),
 ("'s", np.float64(0.0)),
 ("'ve", np.float64(0.0)),
 ('a', np.float64(0.0)),
 ('about', np.float64(0.0)),
 ('above', np.float64(0.0)),
 ('across', np.float64(0.0)),
 ('after', n